## 1. Configurations

### 1-1. Import libraries

In [1]:
from    typing      import  Callable, Hashable, Sequence, Union
from    pathlib     import  Path
import  yaml
import  pickle

from    time            import  time
from    tqdm.notebook   import  trange, tqdm

import  numpy       as      np
import  torch
from    torch.utils.data    import  TensorDataset, DataLoader
import  pandas      as      pd

from    matplotlib.axes     import  Axes
from    matplotlib.figure   import  Figure
import  matplotlib.pyplot   as      plt
import  seaborn             as      sns

import  append_path
from    deep_numerical              import  utils
from    deep_numerical.utils        import  GaussianNormalizer, relative_error
from    deep_numerical              import  numerical   as  dn
from    deep_numerical.numerical    import  distribution    as  D
from    train_utils                 import  load_data, augment_data_2D

from    model_keys      import  *
from    helper_2D       import  *
from    plots           import  *
DIMENSION = 2

dtype  = torch.float
device = torch.device("cuda:2")
BATCH_SIZE      = 1000
METRIC_ORDER    = 2.0

DO_AUGMENTATION = True

DPI = 1000
DPI_PLOT = 100
LINEWIDTH = 0.7
SAVEFIG_PATH = Path().cwd() / f"figures_{DIMENSION}D"
SAVEFIG_PATH.mkdir(exist_ok=True)

### 1-2. Load trained models and training histories

In [2]:
model_data      = load_model_data(f"paths_{DIMENSION}D.yaml")
models          = instantiate_models(model_data, device)
normalizers     = instantiate_normalizers(model_data)
train_histories = instantiate_histories(model_data)

## 2. Conduct test

In [3]:
exp_speed = torch.arange(0, 1.01, 0.05).reshape(-1, 1)
targets = torch.cat(
    [
        torch.load(
            f"further_dataset/Boltzmann_VHS__biMaxwellian__alpha{alpha:.1e}__res064.pth",
            weights_only=False,
        )
        for alpha in exp_speed.flatten()
    ], dim=0
).type(torch.float)
NUM_INSTANCES   = targets.size(0)
NUM_TIME_STEPS  = targets.size(1)
RESOLUTION      = targets.size(-2)
targets = targets.reshape(NUM_INSTANCES, NUM_TIME_STEPS, RESOLUTION, RESOLUTION, 1)


if DO_AUGMENTATION:
    targets = torch.cat([targets.rot90(k, (-3, -2)) for k in range(4)])
    targets = torch.cat([targets, targets.flip(-2)])
    exp_speed = exp_speed.repeat((8, 1))

FileNotFoundError: [Errno 2] No such file or directory: 'further_dataset/Boltzmann_VHS__biMaxwellian__alpha0.0e+00__res064.pth'

In [ ]:
datasets: dict[str, dict[str, TensorDataset]] = {
    k: construct_datasets(k, targets, exp_speed, normalizers[k], 3/utils.LAMBDA, 10)
    for k in models.keys()
}
"""Saves normalized datasets."""
dataloaders: dict[str, DataLoader] = {
    k: DataLoader(v, batch_size=BATCH_SIZE, shuffle=False)
    for k, v in datasets.items()
}
"""Saves normalized dataloaders."""

In [ ]:
print("Test the models")
print("* Initialize the storages of the predictions and the relative errors")
_TensorSet = Union[torch.Tensor, list[torch.Tensor]]
pred_lists:     dict[str, _TensorSet] = {k: [] for k in models.keys()}
"""Saves *decoded* predictions."""
rel_errors:     dict[str, _TensorSet] = {k: [] for k in models.keys()}
"""Saves the relative errors between the *decoded* predictions and targets."""
inference_times:    dict[str, float] = {k: 0.0 for k in models.keys()}


print("* Start evaluating the models")
for k_model, model in models.items():
    print(f"  - Evaluating [{k_model}]...")
    model.eval()
    normalizers[k_model]['data'].to(device)
    normalizers[k_model]['params'].to(device)
    inference_times[k_model] = time()

    with torch.no_grad():
        traj:   torch.Tensor
        params: torch.Tensor
        model_preds = []
        for traj, params in dataloaders[k_model]:
            # Move tensors to the device
            traj    = traj.to(device)
            params  = params.to(device)
            pred    = traj[:, 0]
            # Instantiate a storage for batched predictions
            temp_batch_preds = [
                normalizers[k_model]['data'].decode(traj[:, 0]).cpu()
            ]
            # Compute
            for idx in trange(NUM_TIME_STEPS - 1):
                # Time-marching (in the scaled space) and decoding
                pred = model.forward(pred, params)
                temp_batch_preds.append( normalizers[k_model]['data'].decode(pred).cpu() )
            # Stack predictions for the current batch
            model_preds.append( torch.stack(temp_batch_preds, dim=1) )

    inference_times[k_model] = time() - inference_times[k_model]
    pred_lists[k_model]      = torch.cat(model_preds, dim=0)
    rel_errors[k_model]      = relative_error(pred_lists[k_model], targets, dim=tuple(range(2, 3+DIMENSION)))

print("Done.")

In [ ]:
for k in models.keys():
    rel_error_mean = rel_errors[k].mean(1)
    rel_error_std  = rel_errors[k].std(1)
    print(f"[{k}] Relative error (mean, std)\n\t>>> {float(rel_error_mean[-1]):.2e}, {float(rel_error_std[-1]):.2e}")

In [ ]:
inst_idx = 101
inst_alpha = float(exp_speed[inst_idx])
_model_key = K__RFNO_PARAM

figshape = (RESOLUTION, RESOLUTION)
fig, axes = plt.subplots(3, 6, figsize=(20, 12), dpi=600)
fig.suptitle(
    rf"BiMaxwellian distribution with $\alpha={inst_alpha:.2f}$" + 
    '\n' + 
    f"(model: {_model_key})",
    fontsize=20
)
time_indices = torch.linspace(0, NUM_TIME_STEPS-1, axes.shape[1], dtype=torch.long)

vmaxmin = {'vmax': targets[inst_idx].max().item(), 'vmin': 0}
for col, t_idx in enumerate(map(int, time_indices)):
    _gt     = targets[inst_idx, t_idx].reshape(figshape)
    _pred   = pred_lists[K__RFNO_PARAM][inst_idx, t_idx].reshape(figshape)
    axes[0, col].set_title(
        rf"Ground truth ($t={t_idx*0.1:.1f}$)" + 
        '\n' + 
        f"(error: {rel_errors[_model_key][inst_idx, t_idx].item():.2e})"
    )
    axes[1, col].set_title(rf"Prediction")
    axes[2, col].set_title(rf"Difference")
    fig.colorbar(axes[0, col].imshow(_gt, **vmaxmin))
    fig.colorbar(axes[1, col].imshow(_pred, **vmaxmin))
    fig.colorbar(axes[2, col].imshow((_gt-_pred).abs(), **vmaxmin))
fig.show()

End of file